# 🏆 Phase 1: Team-Aware Smart Miner (Interactive Edition)


Notebook này đã được tách nhỏ thành các bước tương tác. Bạn có thể kiểm tra kết quả của từng Phase một cách trực quan, và có thanh tiến trình theo dõi khi Mining.


## 1. Cài Đặt Thư Viện (Chạy 1 lần)


In [ ]:
!pip install ultralytics opencv-python-headless scikit-learn imagehash scikit-image pandas matplotlib tqdm


## 2. Mount Drive & Import (Chạy 1 lần)


In [ ]:
from google.colab import drive
import os
import cv2
import csv
import imagehash
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from skimage.metrics import structural_similarity as ssim
from ultralytics import YOLO
from tqdm.notebook import tqdm
from IPython.display import display, clear_output

drive.mount('/content/drive')

# --- CẤU HÌNH ĐƯỜNG DẪN ---
PROJECT_ROOT = '/content/drive/MyDrive/Bradford_Bulls_AI'
VIDEO_DIR = os.path.join(PROJECT_ROOT, 'videos')
OUTPUT_FRAMES_DIR = os.path.join(PROJECT_ROOT, 'extracted_dataset')
os.makedirs(OUTPUT_FRAMES_DIR, exist_ok=True)

# --- CẤU HÌNH VIDEO ---
# Sửa tên video của bạn ở đây
VIDEO_NAME = "M06_black_1080p.mp4" 
VIDEO_PATH = os.path.join(VIDEO_DIR, VIDEO_NAME)
OUT_DIR = os.path.join(OUTPUT_FRAMES_DIR, VIDEO_NAME.split('.')[0])
os.makedirs(OUT_DIR, exist_ok=True)
print(f"📁 Video mục tiêu: {VIDEO_PATH}")
print(f"📁 Thư mục lưu kết quả: {OUT_DIR}")



## 3. Lõi Thuật Toán (Chạy để nạp hàm)


In [ ]:
def detect_static_overlays(video_path, num_samples=20):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    samples = []
    step = total_frames // num_samples
    for i in range(num_samples):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i * step)
        ret, frame = cap.read()
        if ret:
            samples.append(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY))
    cap.release()
    
    stack = np.stack(samples, axis=0)
    std_dev = np.std(stack, axis=0)
    
    overlay_mask = (std_dev < 5.0).astype(np.uint8) * 255
    kernel = np.ones((15, 15), np.uint8)
    overlay_mask = cv2.dilate(overlay_mask, kernel, iterations=1)
    
    coverage = np.count_nonzero(overlay_mask) / overlay_mask.size
    print(f"✅ Đã tìm thấy Overlays. Chiếm {coverage*100:.1f}% diện tích màn hình.")
    return overlay_mask

def extract_player_samples(video_path, num_samples=24):
    yolo = YOLO("yolov8n.pt")
    cap = cv2.VideoCapture(video_path)
    crops = []
    frame_idx = 0
    while cap.isOpened() and len(crops) < num_samples:
        frame_idx += 60
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if not ret: break
        results = yolo(frame, classes=[0], conf=0.5, verbose=False)
        for box in results[0].boxes.xyxy.cpu().numpy():
            x1, y1, x2, y2 = map(int, box)
            if (y2 - y1) > 150:
                crops.append(frame[y1:y2, x1:x2])
                if len(crops) >= num_samples: break
    cap.release()
    return crops

def build_team_profile(crops, target_ids):
    target_crops = [crops[i] for i in target_ids]
    hists = []
    for crop in target_crops:
        h, w = crop.shape[:2]
        shirt = crop[int(h*0.1):int(h*0.5), :]
        hsv = cv2.cvtColor(shirt, cv2.COLOR_BGR2HSV)
        hist = cv2.calcHist([hsv], [0, 1], None, [16, 16], [0, 180, 0, 256])
        cv2.normalize(hist, hist, alpha=0, beta=1, norm_type=cv2.NORM_MINMAX)
        hists.append(hist)
    return np.mean(hists, axis=0)

def calculate_team_match_score(crop, target_hist):
    h, w = crop.shape[:2]
    shirt = crop[int(h*0.1):int(h*0.5), :]
    hsv = cv2.cvtColor(shirt, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0, 1], None, [16, 16], [0, 180, 0, 256])
    cv2.normalize(hist, hist, alpha=0, beta=1, norm_type=cv2.NORM_MINMAX)
    distance = cv2.compareHist(target_hist, hist, cv2.HISTCMP_BHATTACHARYYA)
    return 1.0 - distance

class TeamAwareMiner:
    def __init__(self, overlay_mask, target_hist):
        self.yolo = YOLO("yolov8n.pt")
        self.overlay_mask = overlay_mask
        self.target_hist = target_hist
        self.MIN_PLAYER_HEIGHT = 200
        self.MIN_SHARPNESS = 80.0
        self.MIN_TEAM_SCORE = 0.50
        self.PHASH_THRESH = 10
        self.SSIM_THRESH = 0.90
        
    def is_in_overlay(self, box):
        x1, y1, x2, y2 = box
        cy, cx = int((y1+y2)/2), int((x1+x2)/2)
        if cy >= self.overlay_mask.shape[0] or cx >= self.overlay_mask.shape[1]: return False
        return self.overlay_mask[cy, cx] == 255
        
    def check_duplicate(self, frame, prev_frame, prev_hash):
        curr_hash = imagehash.phash(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
        if prev_frame is None: return False, curr_hash
        if (curr_hash - prev_hash) < self.PHASH_THRESH:
            gray1 = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
            gray2 = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            if ssim(gray1, gray2) > self.SSIM_THRESH: return True, curr_hash
        return False, curr_hash



## 4. Phase 0A: Auto-Detect Static Overlays (Scoreboard)


In [ ]:
print("🔍 Đang phân tích Scoreboard (Static Overlays)...")
OVERLAY_MASK = detect_static_overlays(VIDEO_PATH)

plt.figure(figsize=(8,4))
plt.imshow(OVERLAY_MASK, cmap='gray')
plt.title("Detected Static Overlays (White = Ignored Zone)")
plt.axis('off')
plt.show()


## 5. Phase 0B: Semi-Supervised Team Calibration (Lấy mẫu)


In [ ]:
print("🏃 Đang trích xuất ngẫu nhiên 24 cầu thủ từ video...")
PLAYER_CROPS = extract_player_samples(VIDEO_PATH, num_samples=24)

fig, axes = plt.subplots(4, 6, figsize=(15, 10))
for i, ax in enumerate(axes.flat):
    if i < len(PLAYER_CROPS):
        ax.imshow(cv2.cvtColor(PLAYER_CROPS[i], cv2.COLOR_BGR2RGB))
        ax.set_title(f"ID: {i}", color="red", fontweight="bold")
    ax.axis("off")
plt.suptitle("Hãy ghi lại các ID thuộc đội Bradford Bulls!", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()


## 6. Lựa Chọn Đội Nhà


In [ ]:
# Nhập các ID áo đội nhà (cách nhau bởi dấu phẩy, ví dụ: 0, 3, 5)
raw_input = input("👉 Nhập các ID của đội Bradford Bulls: ")
TARGET_TEAM_IDS = [int(x.strip()) for x in raw_input.split(",") if x.strip().isdigit()]

print(f"Bạn đã chọn: {TARGET_TEAM_IDS}")
TARGET_HIST = build_team_profile(PLAYER_CROPS, TARGET_TEAM_IDS)
print("✅ Đã xây dựng xong Histogram màu áo cho đội nhà!")



## 7. Phase 1: Bắt Đầu Đào Dữ Liệu (Frame Mining)


In [ ]:
miner = TeamAwareMiner(OVERLAY_MASK, TARGET_HIST)
cap = cv2.VideoCapture(VIDEO_PATH)
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
ORIGINAL_FPS = cap.get(cv2.CAP_PROP_FPS)
if ORIGINAL_FPS == 0: ORIGINAL_FPS = 30.0

frame_idx = 0
saved_count = 0
prev_frame = None
prev_hash = None

pbar = tqdm(total=TOTAL_FRAMES, desc="Scanning Video")

while cap.isOpened() and saved_count < 2000:
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    if not ret: break
    
    next_skip = 10 # Default scan interval
    
    is_dup, curr_hash = miner.check_duplicate(frame, prev_frame, prev_hash)
    if not is_dup:
        results = miner.yolo(frame, classes=[0], conf=0.4, verbose=False)
        boxes = results[0].boxes.xyxy.cpu().numpy()
        
        for box in boxes:
            x1, y1, x2, y2 = map(int, box)
            if (y2 - y1) > miner.MIN_PLAYER_HEIGHT and not miner.is_in_overlay([x1, y1, x2, y2]):
                crop = frame[y1:y2, x1:x2]
                gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
                sharpness = cv2.Laplacian(gray, cv2.CV_64F).var()
                
                if sharpness > miner.MIN_SHARPNESS:
                    team_score = calculate_team_match_score(crop, TARGET_HIST)
                    if team_score > miner.MIN_TEAM_SCORE:
                        # 🎯 TÌM THẤY FRAME ĐẠT CHUẨN!
                        prev_frame = frame.copy()
                        prev_hash = curr_hash
                        
                        timestamp_sec = frame_idx / ORIGINAL_FPS
                        h = int(timestamp_sec // 3600)
                        m = int((timestamp_sec % 3600) // 60)
                        s = int(timestamp_sec % 60)
                        ts_str = f"{h:02d}h{m:02d}m{s:02d}s" if h > 0 else f"{m:02d}m{s:02d}s"
                        
                        filename = f"{VIDEO_NAME.split('.')[0]}_{frame_idx:06d}_{ts_str}.jpg"
                        cv2.imwrite(os.path.join(OUT_DIR, filename), frame)
                        
                        saved_count += 1
                        next_skip = 60 # Bỏ qua 2s vì đã có frame xịn
                        
                        pbar.set_postfix({'Saved': saved_count})
                        break
                        
    frame_idx += next_skip
    pbar.update(next_skip)

cap.release()
pbar.close()
print(f"\n✅ KHAI THÁC HOÀN TẤT: {saved_count} frames xuất sắc đã lưu tại {OUT_DIR}")

